In [33]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


# **Mlflow and dagshub initialisation**

In [34]:
!pip install -q dagshub mlflow

In [37]:
import dagshub
import mlflow

dagshub.init(repo_owner="luchit22", repo_name="ML-fraud-detection",mlflow=True)
mlflow.set_experiment("adaboost")

Initialized MLflow to track repo "luchit22/ML-fraud-detection"

Repository luchit22/ML-fraud-detection initialized!

2026/05/02 17:05:36 INFO mlflow.tracking.fluent: Experiment with name 'adaboost' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/f85797506cc64b8eb3cb5169f6468e73', creation_time=1777741536841, experiment_id='2', last_update_time=1777741536841, lifecycle_stage='active', name='adaboost', tags={}, trace_location=None, workspace='default'>

In [36]:
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

# **data cleaning**

In [38]:
train_transaction = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv")
train_identity = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv")

train_transaction.columns = train_transaction.columns.str.replace("-", "_")
train_identity.columns = train_identity.columns.str.replace("-", "_")

train = train_transaction.merge(
    train_identity,
    on="TransactionID",
    how="left"
)

print(train.shape)

(590540, 434)


In [39]:
y = train["isFraud"]
X = train.drop(columns=["isFraud"])

In [44]:
missing_ratio = X.isnull().mean()

cols_to_drop = missing_ratio[missing_ratio > 0.90].index.tolist()

X_clean = X.drop(columns=cols_to_drop)

print("Dropped columns with >90% missing:", len(cols_to_drop))

Dropped columns with >90% missing: 0


In [45]:
X_numeric = X_clean.select_dtypes(exclude=["object"]).copy()

print("Numeric shape:", X_numeric.shape)

Numeric shape: (590540, 392)


# **feature engineering**

In [46]:
X_fe = X_numeric.copy()

if "TransactionAmt" in X_clean.columns:
    X_fe["TransactionAmt_log"] = np.log1p(X_clean["TransactionAmt"])

if "TransactionDT" in X_clean.columns:
    X_fe["TransactionDT_days"] = X_clean["TransactionDT"] / (3600 * 24)
    X_fe["TransactionDT_hours"] = X_clean["TransactionDT"] / 3600

X_fe["num_missing_original"] = X_clean.isnull().sum(axis=1)

X_fe = X_fe.fillna(-999)

print("Shape after feature engineering:", X_fe.shape)

Shape after feature engineering: (590540, 396)


In [47]:
X_sample, _, y_sample, _ = train_test_split(
    X_fe,
    y,
    train_size=100000,
    stratify=y,
    random_state=42
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_sample,
    y_sample,
    test_size=0.2,
    stratify=y_sample,
    random_state=42
)

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)

Train: (80000, 396)
Valid: (20000, 396)


# **feature selection**

In [48]:
temp = X_train.copy()
temp["isFraud"] = y_train.values

target_corr = temp.corr()["isFraud"].abs().sort_values(ascending=False)

top_features = target_corr.index[1:101].tolist()

X_train_top = X_train[top_features]
X_valid_top = X_valid[top_features]

print("Top features selected:", len(top_features))

Top features selected: 100


In [49]:
corr_matrix = X_train_top.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr_features = [
    col for col in upper.columns
    if any(upper[col] > 0.90)
]

X_train_final = X_train_top.drop(columns=high_corr_features)
X_valid_final = X_valid_top.drop(columns=high_corr_features)

print("Removed highly correlated features:", len(high_corr_features))
print("Final feature count:", X_train_final.shape[1])

Removed highly correlated features: 95
Final feature count: 5


# **training**

In [50]:
base_tree = DecisionTreeClassifier(
    max_depth=1,
    random_state=42
)

ada_baseline = AdaBoostClassifier(
    estimator=base_tree,
    n_estimators=100,
    learning_rate=0.5,
    random_state=42
)

ada_baseline.fit(X_train_final, y_train)

AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1,
                                                    random_state=42),
                   learning_rate=0.5, n_estimators=100, random_state=42)

In [51]:
train_proba = ada_baseline.predict_proba(X_train_final)[:, 1]
valid_proba = ada_baseline.predict_proba(X_valid_final)[:, 1]

train_auc = roc_auc_score(y_train, train_proba)
valid_auc = roc_auc_score(y_valid, valid_proba)
auc_gap = train_auc - valid_auc

print("Train ROC-AUC:", train_auc)
print("Validation ROC-AUC:", valid_auc)
print("AUC gap:", auc_gap)

Train ROC-AUC: 0.6950639362135373
Validation ROC-AUC: 0.678678534418949
AUC gap: 0.01638540179458836


In [52]:
with mlflow.start_run(run_name="AdaBoost_Baseline"):
    mlflow.log_param("model", "AdaBoost")
    mlflow.log_param("base_estimator", "DecisionTree")
    mlflow.log_param("base_tree_depth", 1)
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("learning_rate", 0.5)
    mlflow.log_param("train_size", 100000)

    mlflow.log_param("missing_strategy", "fillna_-999")
    mlflow.log_param("dropped_missing_columns_threshold", 0.90)
    mlflow.log_param("feature_engineering", "TransactionAmt_log, TransactionDT_days, TransactionDT_hours, num_missing_original")
    mlflow.log_param("feature_selection_1", "target_correlation_top100")
    mlflow.log_param("feature_selection_2", "correlation_filter")
    mlflow.log_param("correlation_threshold", 0.90)
    mlflow.log_param("num_final_features", X_train_final.shape[1])
    mlflow.log_param("removed_high_corr_features", len(high_corr_features))

    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("valid_auc", valid_auc)
    mlflow.log_metric("auc_gap", auc_gap)

    mlflow.sklearn.log_model(ada_baseline, artifact_path="adaboost_baseline")

2026/05/02 17:16:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 17:16:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run AdaBoost_Baseline at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/2/runs/e8daa2d49b0b41048134e7062a79e249
🧪 View experiment at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/2


In [53]:
ada_configs = [
    {
        "name": "Ada_depth1_estimators100_lr05",
        "max_depth": 1,
        "n_estimators": 100,
        "learning_rate": 0.5
    },
    {
        "name": "Ada_depth2_estimators150_lr03",
        "max_depth": 2,
        "n_estimators": 150,
        "learning_rate": 0.3
    },
    {
        "name": "Ada_depth2_estimators200_lr05",
        "max_depth": 2,
        "n_estimators": 200,
        "learning_rate": 0.5
    },
    {
        "name": "Ada_depth3_estimators150_lr02",
        "max_depth": 3,
        "n_estimators": 150,
        "learning_rate": 0.2
    }
]

In [54]:
ada_results = []
best_ada = None
best_valid_auc = 0
best_config = None

for config in ada_configs:
    with mlflow.start_run(run_name=config["name"]):
        base_tree = DecisionTreeClassifier(
            max_depth=config["max_depth"],
            random_state=42
        )

        model = AdaBoostClassifier(
            estimator=base_tree,
            n_estimators=config["n_estimators"],
            learning_rate=config["learning_rate"],
            random_state=42
        )

        model.fit(X_train_final, y_train)

        train_proba = model.predict_proba(X_train_final)[:, 1]
        valid_proba = model.predict_proba(X_valid_final)[:, 1]

        train_auc = roc_auc_score(y_train, train_proba)
        valid_auc = roc_auc_score(y_valid, valid_proba)
        auc_gap = train_auc - valid_auc

        mlflow.log_param("model", "AdaBoost")
        mlflow.log_param("base_estimator", "DecisionTree")
        mlflow.log_param("max_depth", config["max_depth"])
        mlflow.log_param("n_estimators", config["n_estimators"])
        mlflow.log_param("learning_rate", config["learning_rate"])
        mlflow.log_param("train_size", 100000)

        mlflow.log_param("missing_strategy", "fillna_-999")
        mlflow.log_param("feature_engineering", True)
        mlflow.log_param("feature_selection_1", "target_correlation_top100")
        mlflow.log_param("feature_selection_2", "correlation_filter")
        mlflow.log_param("correlation_threshold", 0.90)
        mlflow.log_param("num_final_features", X_train_final.shape[1])

        mlflow.log_metric("train_auc", train_auc)
        mlflow.log_metric("valid_auc", valid_auc)
        mlflow.log_metric("auc_gap", auc_gap)

        ada_results.append({
            **config,
            "train_auc": train_auc,
            "valid_auc": valid_auc,
            "auc_gap": auc_gap
        })

        if valid_auc > best_valid_auc:
            best_valid_auc = valid_auc
            best_ada = model
            best_config = config

ada_results_df = pd.DataFrame(ada_results).sort_values("valid_auc", ascending=False)
ada_results_df

🏃 View run Ada_depth1_estimators100_lr05 at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/2/runs/76cb9e103ea74aa586f1bdcf63e879e6
🧪 View experiment at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/2
🏃 View run Ada_depth2_estimators150_lr03 at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/2/runs/9b7d4de9be74429d97203b2f3ea576e7
🧪 View experiment at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/2
🏃 View run Ada_depth2_estimators200_lr05 at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/2/runs/8a2d8ca8f9f04c7a8c1818ed50446345
🧪 View experiment at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/2
🏃 View run Ada_depth3_estimators150_lr02 at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/2/runs/3407beb97cbc4cf78e48a9cfab37149c
🧪 View experiment at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/2


,name,max_depth,n_estimators,learning_rate,train_auc,valid_auc,auc_gap
3,Ada_depth3_estimators150_lr02,3,150,0.2,0.701366,0.683264,0.018102
1,Ada_depth2_estimators150_lr03,2,150,0.3,0.698577,0.681432,0.017144
2,Ada_depth2_estimators200_lr05,2,200,0.5,0.697672,0.680865,0.016807
0,Ada_depth1_estimators100_lr05,1,100,0.5,0.695064,0.678679,0.016385


In [55]:
with mlflow.start_run(run_name="AdaBoost_BEST_Model"):
    mlflow.log_param("model", "AdaBoost")
    mlflow.log_param("base_estimator", "DecisionTree")
    mlflow.log_params(best_config)

    mlflow.log_param("missing_strategy", "fillna_-999")
    mlflow.log_param("feature_engineering", True)
    mlflow.log_param("feature_selection", "target_correlation_top100 + correlation_filter")
    mlflow.log_param("num_final_features", X_train_final.shape[1])

    mlflow.log_metric("best_valid_auc", best_valid_auc)

    mlflow.sklearn.log_model(best_ada, artifact_path="adaboost_best_model")

print("Best AdaBoost config:", best_config)
print("Best validation AUC:", best_valid_auc)

2026/05/02 17:19:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 17:19:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run AdaBoost_BEST_Model at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/2/runs/02e6d4a0dada43d5acbedd5483a7443c
🧪 View experiment at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/2
Best AdaBoost config: {'name': 'Ada_depth3_estimators150_lr02', 'max_depth': 3, 'n_estimators': 150, 'learning_rate': 0.2}
Best validation AUC: 0.6832644337527758
